# Bone fracture detection on FracAtlas
CNN vs ScatNet (+ ResNet18), six XAI methods, fracture boxes, guided training and a YOLO pipeline.
All code is in `src/`; run it on Kaggle with `./run.sh push`.

## Setup

In [ ]:
# Setup: on Kaggle ./run.sh push unpacks the code into /tmp/bfd; locally this repository is used
import sys
from pathlib import Path

ROOT = next(p for p in (Path("/tmp/bfd"), Path.cwd().parent, Path.cwd()) if (p / "src").is_dir())
sys.path.insert(0, str(ROOT))
from src.bootstrap import setup
paths = setup(ROOT)   # installs kymatio, captum, ultralytics if missing; finds the dataset; makes the output folders

## Settings

In [ ]:
QUICK      = False     # True: 5-minute health check on a small subset
TRAIN      = True      # False: reload a previous run from RESULTS_DIR
MODELS     = ["cnn", "scatnet", "resnet18"]
K_FOLDS    = 5
EPOCHS     = 20
BATCH_SIZE = 32
LR         = {"cnn": 1e-3, "scatnet": 1e-3, "resnet18": 1e-4}
SELECT_BY  = "f1"      # CV metric that picks the best model
DUP_THRESHOLD = 0.97   # thumbnail similarity of two copies of one X-ray
XAI_PER_CLASS = 5      # test X-rays explained per class
XAI_PARAMS = dict(ig_steps=64, window=16, stride=8, lime_samples=1000)
BOX_THRESHOLD = 0.5    # heatmap -> box: pixels above this share of the maximum
GUIDE      = dict(box=1.0, contrast=0.1, warmup=3, temperature=0.1, margin=8)   # guided training
YOLO_PARAMS = dict(model="yolov8s.pt", epochs=50, imgsz=640, batch=16)
YOLO_CONF  = 0.25
SEED = 42

MAX_PER_CLASS = None
if QUICK:
    MAX_PER_CLASS, K_FOLDS, EPOCHS, XAI_PER_CLASS = 60, 2, 1, 2
    XAI_PARAMS = dict(ig_steps=8, window=32, stride=16, lime_samples=50)
    GUIDE = dict(GUIDE, warmup=0)
    YOLO_PARAMS = dict(model="yolov8n.yaml", epochs=1, imgsz=320, batch=8)

import numpy as np, pandas as pd
from src import data, models, training, xai, localize, guidance, detect, pipeline, plots, report, utils

DATA_DIR, RESULTS_DIR, FIG_DIR, CACHE_DIR = paths["data"], paths["results"], paths["figures"], paths["cache"]
utils.set_seed(SEED)
device = utils.get_device()
NUM_WORKERS = 3 if device.type == "cuda" else 0
print("device:", device, "| quick run" if QUICK else "| full run")

## 1. Data

In [ ]:
dataset = data.load_dataset(DATA_DIR, cache_dir=CACHE_DIR, max_per_class=MAX_PER_CLASS, seed=SEED)
class_names = dataset.class_names   # 0 = fractured, 1 = not fractured
print(f"{len(dataset)} X-rays, {(dataset.labels == 0).sum()} fractured, {sum(map(len, dataset.boxes))} boxes")

In [ ]:
# copies of the same X-ray (flipped, resized, brighter) must stay in one split
dup = data.study_duplicates(dataset, DUP_THRESHOLD, device=device)
plots.show_duplicate_pairs(dataset, dup["pairs"], title="Copies of the same X-ray", save_to=FIG_DIR / "duplicates.png")
dup["table"]

In [ ]:
# stratified ~70/15/15 split, copies kept together
split = data.split_indices(dataset.labels, dup["groups"], seed=SEED)
sets = {name: dataset.subset(idx) for name, idx in split.items()}
train_groups = dup["groups"][split["train"]]
data.summary_table(sets)

In [ ]:
plots.plot_dataset_overview(sets, save_to=FIG_DIR / "dataset_overview.png")
plots.show_samples(sets["train"], title="Training X-rays (dashed: fracture boxes)", save_to=FIG_DIR / "samples_train.png")

## 2. Models (same classifier, only the input size differs)

In [ ]:
layouts = {name: models.classifier_layout(models.build_model(name, pretrained=False)) for name in MODELS}
assert all(layout == layouts["cnn"] for layout in layouts.values()), "the classifiers differ"
models.summary(MODELS)

## 3. Cross-validation (train split)

In [ ]:
cv = {}
for name in MODELS:
    if TRAIN:
        cv[name] = training.cross_validate(name, sets["train"], train_groups, device, K_FOLDS, EPOCHS, BATCH_SIZE,
                                           LR[name], seed=SEED, num_workers=NUM_WORKERS, results_dir=RESULTS_DIR)
    else:
        cv[name] = training.load_json(RESULTS_DIR / "cv" / f"{name}.json")
training.cv_table(cv)   # mean ± std, %

## 4. Final training and test

In [ ]:
trained, final = {}, {}
for name in MODELS:
    if TRAIN:
        trained[name], final[name] = training.train_final(name, sets["train"], sets["val"], device, EPOCHS, BATCH_SIZE,
                                                          LR[name], seed=SEED, num_workers=NUM_WORKERS,
                                                          results_dir=RESULTS_DIR)
    else:
        trained[name] = training.load_model(name, RESULTS_DIR, device)
        final[name] = training.load_json(RESULTS_DIR / "final" / f"{name}.json")
plots.plot_learning_curves(cv, final, save_to=FIG_DIR / "learning_curves.png")

In [ ]:
test = {name: training.evaluate_test(trained[name], sets["test"], device) for name in MODELS}
training.test_table(test)

In [ ]:
plots.plot_confusion_matrices(test, class_names, save_to=FIG_DIR / "confusion_matrices.png")
plots.plot_roc(test, save_to=FIG_DIR / "roc.png")
mcnemar = training.mcnemar_table(test)   # is the gap between two models real?
mcnemar

## 5. Best model (chosen by cross-validation)

In [ ]:
comparison = training.comparison_table(cv, test, final)
BEST = training.select_best(cv, SELECT_BY)
GUIDE_MODEL = BEST if BEST != "scatnet" else max((n for n in cv if n != "scatnet"), key=lambda n: cv[n]["mean"][SELECT_BY])
plots.plot_model_comparison(comparison, best=BEST, save_to=FIG_DIR / "model_comparison.png")
comparison

In [ ]:
errors = training.error_table(test, class_names)
print(errors.attrs["wrong for every model"], "test X-rays are wrong for every model")
errors

## 6. Filters: learned (CNN) vs fixed (ScatNet)

In [ ]:
plots.plot_conv_filters(trained["cnn"].first_conv.weight, "CNN: 32 learned 7x7 filters", save_to=FIG_DIR / "filters_cnn.png")
plots.plot_scattering_filters(trained["scatnet"].J, trained["scatnet"].L, save_to=FIG_DIR / "filters_scatnet.png")
plots.plot_frequency_coverage(trained["cnn"].first_conv.weight, trained["scatnet"].J, trained["scatnet"].L,
                              save_to=FIG_DIR / "frequency_coverage.png")
if "resnet18" in trained:
    plots.plot_conv_filters(trained["resnet18"].first_conv.weight, "ResNet18: first 32 filters",
                            save_to=FIG_DIR / "filters_resnet18.png")

## 7. Six XAI methods (2 X-rays per class shown)

In [ ]:
XAI_MODELS = list(dict.fromkeys(["cnn", "scatnet", BEST, GUIDE_MODEL]))   # + the model guided in section 11
correct = np.all([np.array(test[n]["y_pred"]) == sets["test"].labels for n in XAI_MODELS], axis=0)
xai_idx = data.pick_images(sets["test"].labels, XAI_PER_CLASS, seed=SEED, allowed=correct)
xai_images, xai_labels = sets["test"].tensors(xai_idx)
n_frac = int((xai_labels == 0).sum())
SHOW = [0, 1, n_frac, n_frac + 1]   # 2 fractured + 2 not fractured
xai.applicability({name: trained[name] for name in XAI_MODELS})

In [ ]:
attributions, seconds = {}, {}
for name in XAI_MODELS:
    attributions[name], seconds[name] = xai.explain_all(
        trained[name], xai_images, xai_labels, methods={**xai.METHODS, **xai.SCRATCH},
        cache=RESULTS_DIR / "attributions" / f"{name}.npz", recompute=TRAIN, **XAI_PARAMS)

In [ ]:
for name in XAI_MODELS:
    plots.plot_attribution_grid(xai_images, xai_labels, attributions[name], class_names, methods=xai.METHODS,
                                indices=SHOW, title=f"{plots.label(name)}: six XAI methods", save_to=FIG_DIR / f"xai_{name}.png")

## 8. Occlusion from scratch vs Captum

In [ ]:
scratch = xai.scratch_vs_captum(attributions)
plots.plot_scratch_vs_captum(xai_images, attributions, index=SHOW[0], save_to=FIG_DIR / "occlusion_scratch_vs_captum.png")
scratch

## 9. Faithfulness (deletion test) and agreement

In [ ]:
curves = {name: xai.deletion_curves(trained[name], xai_images, xai_labels, attributions[name]) for name in XAI_MODELS}
deletion = {name: xai.deletion_table(curves[name]) for name in XAI_MODELS}
plots.plot_deletion_curves(curves, save_to=FIG_DIR / "deletion_curves.png")
pd.DataFrame({plots.label(n): d["deletion AUC (mean)"] for n, d in deletion.items()})   # lower = better

In [ ]:
plots.plot_agreement({name: xai.agreement_matrix(attributions[name]) for name in XAI_MODELS},
                     save_to=FIG_DIR / "xai_agreement.png")
pd.DataFrame(seconds).round(3)   # seconds per X-ray

## 10. From heatmap to box (hit rate on the fractured test X-rays)

In [ ]:
frac = np.flatnonzero(sets["test"].labels == 0)
box_images, _ = sets["test"].tensors(frac)
true_boxes = [sets["test"].boxes[i] for i in frac]
box_maps = {name: xai.explain_all(trained[name], box_images, np.zeros(len(frac), dtype=int),   # explain "fractured"
                                  cache=RESULTS_DIR / "attributions" / f"{name}_boxes.npz", recompute=TRAIN,
                                  **XAI_PARAMS)[0] for name in XAI_MODELS}
localization = {name: localize.localization_table(box_maps[name], true_boxes, BOX_THRESHOLD) for name in XAI_MODELS}
pd.concat({plots.label(n): t for n, t in localization.items()}, axis=1)   # higher = better

In [ ]:
for name in XAI_MODELS:
    plots.plot_fracture_boxes(box_images[:3], box_maps[name], true_boxes[:3], methods=xai.METHODS, threshold=BOX_THRESHOLD,
                              title=f"{plots.label(name)}: where is the fracture?", save_to=FIG_DIR / f"boxes_{name}.png")

## 11. Guided training: box loss + contrastive loss

In [ ]:
# A = the plain model (BEST, or the best CNN-type model if ScatNet won), B = + box loss, C = + box + contrastive loss
GUIDED = {f"{GUIDE_MODEL}_box": dict(GUIDE, contrast=0.0), f"{GUIDE_MODEL}_box_contrast": GUIDE}
for key, params in GUIDED.items():
    if TRAIN:
        trained[key], final[key] = guidance.train_guided(GUIDE_MODEL, key, sets["train"], sets["val"], device, epochs=EPOCHS,
                                                         batch_size=BATCH_SIZE, lr=LR[GUIDE_MODEL], seed=SEED,
                                                         num_workers=NUM_WORKERS, results_dir=RESULTS_DIR, **params)
    else:
        trained[key] = training.load_model(key, RESULTS_DIR, device, arch=GUIDE_MODEL)
        final[key] = training.load_json(RESULTS_DIR / "final" / f"{key}.json")
    test[key] = training.evaluate_test(trained[key], sets["test"], device)

In [ ]:
# the same XAI checks for the guided models
for key in GUIDED:
    box_maps[key] = xai.explain_all(trained[key], box_images, np.zeros(len(frac), dtype=int),
                                    cache=RESULTS_DIR / "attributions" / f"{key}_boxes.npz", recompute=TRAIN, **XAI_PARAMS)[0]
    localization[key] = localize.localization_table(box_maps[key], true_boxes, BOX_THRESHOLD)
    attributions[key], seconds[key] = xai.explain_all(trained[key], xai_images, xai_labels,
                                                     cache=RESULTS_DIR / "attributions" / f"{key}.npz", recompute=TRAIN, **XAI_PARAMS)
    deletion[key] = xai.deletion_table(xai.deletion_curves(trained[key], xai_images, xai_labels, attributions[key]))

In [ ]:
VARIANTS = [GUIDE_MODEL, *GUIDED]
plots.plot_localization({k: localization[k] for k in VARIANTS}, save_to=FIG_DIR / "guided_localization.png")
plots.plot_fracture_boxes(box_images[:3], box_maps[VARIANTS[-1]], true_boxes[:3], methods=xai.METHODS, threshold=BOX_THRESHOLD,
                          title=f"{plots.label(VARIANTS[-1])}: where is the fracture?", save_to=FIG_DIR / f"boxes_{VARIANTS[-1]}.png")
guided = guidance.comparison(VARIANTS, test, localization, deletion, {k: plots.label(k) for k in VARIANTS})
guided

## 12. YOLO and the full pipeline

In [ ]:
# YOLOv8 learns the fracture boxes of the training X-rays (separate from the classifiers)
yolo_data = detect.write_yolo_dataset(sets, CACHE_DIR / "yolo")
detector = (detect.train(yolo_data, RESULTS_DIR / "detector", seed=SEED, **YOLO_PARAMS) if TRAIN
            else detect.load(RESULTS_DIR / "detector"))
detector_scores = detect.box_metrics(detector, yolo_data, imgsz=YOLO_PARAMS["imgsz"])   # FracAtlas paper: mAP@0.5 0.562
detector_scores

In [ ]:
detections = detect.find_boxes(detector, sets["test"].paths, data.IMAGE_SIZE, conf=YOLO_CONF, imgsz=YOLO_PARAMS["imgsz"])
detector_boxes = localize.score_boxes([detections[i] for i in frac], true_boxes, box_images.shape[-2:])
plots.plot_localization(localization, detector=detector_boxes["hit rate (%)"], save_to=FIG_DIR / "localization.png")
localize.detector_vs_xai(localization, detector_boxes, {k: plots.label(k) for k in localization})

In [ ]:
# best model and XAI method chosen on the val split; verdict: agree -> decide, disagree -> needs review
PIPE_MODEL = guidance.best_on_val(final, VARIANTS)
BOX_METHOD, val_hits = pipeline.pick_method(trained[PIPE_MODEL], sets["val"], BOX_THRESHOLD, **XAI_PARAMS)
print(f"classifier: {plots.label(PIPE_MODEL)} | explanation: {BOX_METHOD}")
cases = pipeline.run(trained[PIPE_MODEL], sets["test"], test[PIPE_MODEL]["prob_positive"], detections, BOX_METHOD, device,
                     **XAI_PARAMS)
pipeline.outcomes(cases)

In [ ]:
plots.plot_pipeline_examples(sets["test"], cases, save_to=FIG_DIR / "pipeline_examples.png")
print("\n\n".join(cases["report"].head(3)))
pipeline.scores(cases)

## 13. Save the results

In [ ]:
settings = dict(models=MODELS, k_folds=K_FOLDS, epochs=EPOCHS, batch_size=BATCH_SIZE, lr=LR, seed=SEED, quick=QUICK,
                xai_per_class=XAI_PER_CLASS, xai_params=XAI_PARAMS, box_threshold=BOX_THRESHOLD, guide=GUIDE, yolo=YOLO_PARAMS)
summary = report.export(RESULTS_DIR, settings=settings, dataset=data.summary_table(sets), duplicates=dup["table"], cv=cv,
                        final=final, test=test, best=BEST, select_by=SELECT_BY, mcnemar=mcnemar, deletion=deletion,
                        seconds=seconds, scratch=scratch, localization=localization, guided=guided,
                        pipeline=dict(classifier=PIPE_MODEL, box_method=BOX_METHOD, detector=detector_scores,
                                      detector_boxes=detector_boxes, cases=pipeline.scores(cases)))
report.key_findings(summary);

## Recap
- **Data**: FracAtlas, copies of an X-ray kept together, 1 in 6 fractured, so the loss is weighted.
- **Models**: CNN vs ScatNet with the same classifier, ResNet18 as reference; the best by cross-validation F1.
- **XAI**: six methods, scored by the deletion test (faithful?) and the hit rate (on the fracture?); our Occlusion = Captum.
- **Guided training**: the best model also learns *where* the fracture is (box loss + contrastive loss); compared with the plain model.
- **Pipeline**: classifier + best XAI + YOLO; they agree, it decides; they disagree, *needs review*.
- The numbers are printed just above.